# RedPill Verification Process

This notebook walks through the non-streaming example from `signature_verifier.py` one step at a time.

In [70]:
import json
import os
import secrets
from hashlib import sha256

import requests
from dotenv import load_dotenv
from eth_account import Account
from eth_account.messages import encode_defunct

from attestation_verifier import (
    check_gpu,
    check_report_data,
    check_tdx_quote,
    show_sigstore_provenance,
)

load_dotenv()

API_KEY = os.environ.get("API_KEY", "")
BASE_URL = "https://api.redpill.ai"
MODEL = "phala/qwen3-vl-30b-a3b-instruct"

if not API_KEY:
    raise ValueError("Set API_KEY in your environment or .env before running the notebook.")


def sha256_text(text):
    return sha256(text.encode()).hexdigest()


def recover_signer(text, signature):
    message = encode_defunct(text=text)
    return Account.recover_message(message, signature=signature)


## Step 1: Chat via API

In [71]:
body = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Hello, what is the capital of France?"}],
    "stream": False,
    "max_tokens": 1000,
}
body_json = json.dumps(body)

print("Sending request:")
print(json.dumps(body, indent=2))

response = requests.post(
    f"{BASE_URL}/v1/chat/completions",
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    },
    data=body_json,
    timeout=30,
)
response.raise_for_status()

payload = response.json()
chat_id = payload["id"]
response_text = response.text

print("Chat ID:", chat_id)
print("API response:")
print(json.dumps(payload, indent=2))


Sending request:
{
  "model": "phala/qwen3-vl-30b-a3b-instruct",
  "messages": [
    {
      "role": "user",
      "content": "Hello, what is the capital of France?"
    }
  ],
  "stream": false,
  "max_tokens": 1000
}
Chat ID: chatcmpl-a8c9225bb166fd4b
API response:
{
  "id": "chatcmpl-a8c9225bb166fd4b",
  "object": "chat.completion",
  "created": 1774904423,
  "model": "qwen/qwen3-vl-30b-a3b-instruct",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "The capital of France is Paris.",
        "refusal": null,
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": [],
        "reasoning": null,
        "reasoning_content": null
      },
      "logprobs": null,
      "finish_reason": "stop",
      "stop_reason": null,
      "token_ids": null
    }
  ],
  "service_tier": null,
  "system_fingerprint": null,
  "usage": {
    "prompt_tokens": 17,
    "total_tokens": 25,
    "completio

## Step 2.1: Get TEE's signature on (request, response) pair

In [72]:
signature_response = requests.get(
    f"{BASE_URL}/v1/signature/{chat_id}?model={MODEL}",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30,
)
signature_response.raise_for_status()
signature_payload = signature_response.json()

print(json.dumps(signature_payload, indent=2))


{
  "text": "4a9e8696c301f3218e0ff937240c111f3480a1a10aef03526be4b722f3802ea1:fc573871b873967d3113e3aaaf49d7cfdfec4a81dca8d2b3ba0568cd1c9fdcd8",
  "signature": "0xfdb041031435bf40b9f1a26acc384d9644a58404a7f8c3541ade3c186d5b1102457b3697115c13c4226da3f9077dd6f22cdf72a504bc37b91b287d197fea61af1b",
  "signing_address": "0xE3eB249b37Eaed1E21E4FE4eA0B80C6D231408E6",
  "signing_algo": "ecdsa"
}


## Step 2.2: Verify the sig

In [73]:
request_hash = sha256_text(body_json)
response_hash = sha256_text(response_text)

hashed_text = signature_payload["text"]
request_hash_server, response_hash_server = hashed_text.split(":")

print("Request hash matches:", request_hash == request_hash_server)
print("Response hash matches:", response_hash == response_hash_server)

signature = signature_payload["signature"]
signing_address = signature_payload["signing_address"]
recovered = recover_signer(hashed_text, signature)

# print("Claimed signing address:", signing_address)
print("Recovered signing address:", recovered)
print("Signature valid:", recovered.lower() == signing_address.lower())


Request hash matches: True
Response hash matches: False
Recovered signing address: 0xE3eB249b37Eaed1E21E4FE4eA0B80C6D231408E6
Signature valid: True


## Step 3: Verify the signer runs in TDX TEE

### Step 3.1: Fetch a fresh attestation for the signer

In [74]:
nonce = secrets.token_hex(32)
attestation_response = requests.get(
    f"{BASE_URL}/v1/attestation/report?model={MODEL}&nonce={nonce}&signing_address={signing_address}",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30,
)
attestation_response.raise_for_status()
report = attestation_response.json()

if "all_attestations" in report:
    attestation = next(
        item for item in report["all_attestations"]
        if item["signing_address"].lower() == signing_address.lower()
    )
else:
    attestation = report

with open("attestation-report.json", "w", encoding="utf-8") as file:
    json.dump(attestation, file, indent=2)


print("Has Intel quote:", "intel_quote" in attestation)
print("Has NVIDIA payload:", "nvidia_payload" in attestation)

Has Intel quote: True
Has NVIDIA payload: True


### Step 3.2: Verify Intel TDX quote & Check that report data binds the signer and nonce

`check_tdx_quote(attestation)` does three things:

1. It takes the raw `intel_quote` field from the attestation we fetched from Redpill.
2. It sends that quote to Phala's TDX verification endpoint at `https://cloud-api.phala.network/api/v1/attestations/verify`.
3. It reads the verifier response and prints whether the quote was marked as verified, along with any message returned by the verifier.

The returned `intel_result` object also contains the decoded quote body, which is useful for teaching because it exposes measurements like `mrtd`, `mrconfig`, and `reportdata` that we inspect in the next steps.

In [75]:
intel_result = check_tdx_quote(attestation)

quote_body = intel_result["quote"]["body"]
print("mr_td:", quote_body.get("mrtd"))
print("mr_config:", quote_body.get("mrconfig"))
print("report_data:", quote_body.get("reportdata"))

report_data_check = check_report_data(attestation, nonce, intel_result)

Intel TDX quote verified: True
mr_td: 0xb24d3b24e9e3c16012376b52362ca09856c4adecb709d5fac33addf1c47e193da075b125b6c364115771390a5461e217
mr_config: 0x016e35f7b4a18a41c77ee0288b27139cbb62ebb5a6ab744e2fa1bfc44d08a9e864000000000000000000000000000000
report_data: 0xe3eb249b37eaed1e21e4fe4ea0b80c6d231408e6000000000000000000000000086e781e50292a57c3e2b32e4b9b19fe0955747310b0c83a454a31638d97436d
Signing algorithm: ecdsa
Report data binds signing address: True
Report data embeds request nonce: True


## Step 6: Verify the GPU attestation

`check_gpu(attestation, nonce)` follows the GPU evidence path in `attestation_verifier.py`:

1. It reads the `nvidia_payload` field from the attestation and parses it as JSON.
2. It compares the GPU payload's `nonce` against the nonce we generated when we requested the attestation. This checks freshness and helps prevent replay.
3. It sends the whole GPU evidence payload to NVIDIA's NRAS verifier at `https://nras.attestation.nvidia.com/v3/attest/gpu`.
4. It extracts the JWT token from NVIDIA's response.
5. It decodes the JWT payload and reads the `x-nvidia-overall-att-result` field, which is the overall verdict for the GPU attestation.

So this step teaches two different ideas at once: first, the attestation bundle is tied to our nonce, and second, NVIDIA's verifier independently confirms whether the GPU evidence is valid.

In [76]:
gpu_check = check_gpu(attestation, nonce)
gpu_check


GPU payload nonce matches request_nonce: True
NVIDIA attestation verdict: True


{'nonce_matches': True, 'verdict': True}

## Step 7: Check Sigstore provenance for the attested containers

`show_sigstore_provenance(attestation)` explains the software supply-chain side of the attestation:

1. It looks inside `attestation["info"]["tcb_info"]` to find the `app_compose` data.
2. It scans that compose content for container image digests that match the `@sha256:...` pattern.
3. It deduplicates those digests so the same image is only checked once.
4. For each digest, it builds a Sigstore search URL like `https://search.sigstore.dev/?hash=sha256:...`.
5. It sends an HTTP `HEAD` request to each URL and reports whether the provenance entry is accessible.

This step does not prove the enclave hardware is genuine by itself. Instead, it helps teach how the attestation points to the exact container images used by the workload, and how those images can be traced back through Sigstore provenance records.

In [77]:
print(json.dumps(attestation["info"]["tcb_info"], indent=2))

{
  "mrtd": "b24d3b24e9e3c16012376b52362ca09856c4adecb709d5fac33addf1c47e193da075b125b6c364115771390a5461e217",
  "rtmr0": "6ffe4a2c12f07eccb857f70f370a5af848a7062905cd95adc43abb1f62c39e330aa3c8aeb8f162656c025f3f527600f1",
  "rtmr1": "6e1afb7464ed0b941e8f5bf5b725cf1df9425e8105e3348dca52502f27c453f3018a28b90749cf05199d5a17820101a7",
  "rtmr2": "54a76ae236a9ab1699379c54d70252cd8e6d20ae398fd8cdb240e39bf3a51074aeeee255a375772b8b431421a5435a0d",
  "rtmr3": "ab840652287ad07b16defc22b4f34b9dc00c0e8cf0bb42380c1c4613dd3f27a3a9e9d4fd8b1b87038f10e243de650db4",
  "app_compose": "{\"allowed_envs\":[\"TOKEN\",\"HF_TOKEN\",\"CLOUDFLARE_API_TOKEN\",\"DSTACK_GATEWAY_DOMAIN\",\"CERTBOT_EMAIL\",\"REDIS_PASSWORD\",\"DSTACK_AUTHORIZED_KEYS\"],\"docker_compose_file\":\"x-common: &common-config\\n  restart: always\\n  logging:\\n    driver: \\\"json-file\\\"\\n    options:\\n      max-size: \\\"100m\\\"\\n      max-file: \\\"5\\\"\\n  runtime: nvidia\\n  environment:\\n    - NVIDIA_VISIBLE_DEVICES=all\\n    

In [79]:
import re
def check_sigstore_links(links):
    """Check that Sigstore links are accessible (not 404)."""
    results = []
    for link in links:
        try:
            response = requests.head(link, timeout=10, allow_redirects=True)
            accessible = response.status_code < 400
            results.append((link, accessible, response.status_code))
        except requests.RequestException as e:
            results.append((link, False, str(e)))
    return results
def extract_sigstore_links(compose):
    SIGSTORE_SEARCH_BASE = "https://search.sigstore.dev/?hash="
    """Extract all @sha256:xxx image digests and return Sigstore search links."""
    if not compose:
        return []

    # Match @sha256:hexdigest pattern in Docker compose
    pattern = r'@sha256:([0-9a-f]{64})'
    digests = re.findall(pattern, compose)

    # Deduplicate digests while preserving order
    seen = set()
    unique_digests = []
    for digest in digests:
        if digest not in seen:
            seen.add(digest)
            unique_digests.append(digest)

    return [f"{SIGSTORE_SEARCH_BASE}sha256:{digest}" for digest in unique_digests]

def show_sigstore_provenance(attestation):
    """Extract and display Sigstore provenance links from attestation."""
    print("here is the attestation info:")
    tcb_info = attestation.get("info", {}).get("tcb_info", {})
    if isinstance(tcb_info, str):
        tcb_info = json.loads(tcb_info)
    compose = tcb_info.get("app_compose")
    if not compose:
        print("\nNo app_compose found in attestation, skipping Sigstore provenance check.")
        return

    sigstore_links = extract_sigstore_links(compose)
    if not sigstore_links:
        print("\nNo container image digests found in compose, skipping Sigstore provenance check.")
        return

    print("\n🔐 Sigstore provenance")
    print("Checking Sigstore accessibility for container images...")
    link_results = check_sigstore_links(sigstore_links)

    for link, accessible, status in link_results:
        if accessible:
            print(f"  ✓ {link} (HTTP {status})")
        else:
            print(f"  ✗ {link} (HTTP {status})")

show_sigstore_provenance(attestation)
print('end=')

tcb_info = attestation["info"]["tcb_info"]
if isinstance(tcb_info, str):
    tcb_info = json.loads(tcb_info)

app_compose = tcb_info["app_compose"]
if isinstance(app_compose, str):
    app_compose = json.loads(app_compose)

# print(app_compose["docker_compose_file"])

here is the attestation info:

No container image digests found in compose, skipping Sigstore provenance check.
end=
